In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import scipy
import math
from sko.GA import GA
from scipy.linalg import expm
from scipy.optimize import minimize

from pymoo.core.problem import Problem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM

import notears.linear

In [2]:
LAMBDA = 0.1

In [ ]:
def generate_sem(n, noise="gaussian", seed=42):

    rng = np.random.default_rng(seed)

    X1 = rng.normal(size=n)
    X2 = rng.normal(size=n)

    if noise == "gaussian":
        e3 = rng.normal(size=n)
        e4 = rng.normal(size=n)

    elif noise == "laplace":
        e3 = rng.laplace(size=n)
        e4 = rng.laplace(size=n)

    X3 = 1.0 * X1 + 0.8 * X2 + e3
    X4 = 1.2 * X3 + e4

    return pd.DataFrame({
        "X1": X1,
        "X2": X2,
        "X3": X3,
        "X4": X4
    })

In [4]:
dataset = generate_sem(10000)
X = dataset.values
n = X.shape[0]
dim = X.shape[1]

In [5]:
def acyclic_restriction(W):
    W = np.asarray(W).reshape(dim, dim)

    W_squared = W * W

    return np.trace(expm(W_squared)) - dim


# ============================================================
# Função objetivo
# ============================================================

def augmented_lagrangian(W, rho, alpha):

    W = np.asarray(W).reshape(dim, dim)

    R = X - X @ W

    reconstruction = (
        1 / (2 * n)
        * (R ** 2).sum()
    )

    l1_pen = LAMBDA * np.abs(W).sum()

    h = acyclic_restriction(W)

    return (
        reconstruction
        + l1_pen
        + (rho / 2) * h**2
        + alpha * h
    )

In [6]:
class NOTEARSLagrangianProblem(Problem):

    def __init__(self, rho, alpha):

        self.rho = rho
        self.alpha = alpha

        super().__init__(
            n_var=dim * dim,
            n_obj=1,
            xl=-3.0,
            xu=3.0
        )

    def _evaluate(self, W, out, *args, **kwargs):

        F = np.zeros(len(W))

        for i in range(len(W)):

            F[i] = augmented_lagrangian(
                W[i],
                self.rho,
                self.alpha
            )

        out["F"] = F.reshape(-1, 1)

In [7]:
algorithm = GA(

    pop_size=100,

    crossover=SBX(
        prob=0.7,
        eta=20
    ),

    mutation=PM(
        prob=1.0 / (dim * dim),
        eta=15
    ),

    eliminate_duplicates=True
)

In [8]:
rho = 1.0
alpha = 0.0

W_est = None

h_previous = np.inf

for outer_iter in range(10):

    print(f"\n===== Iteração externa {outer_iter + 1} =====")

    problem = NOTEARSLagrangianProblem(
        rho=rho,
        alpha=alpha
    )
    

    res = minimize(
        problem,
        algorithm,
        termination=("n_gen", 1000),
        sampling=W_est if W_est is not None else FloatRandomSampling(),
        seed=42 + outer_iter,
        verbose=False
    )

    W_est = res.X.reshape(dim, dim)

    h = acyclic_restriction(W_est)

    objective = augmented_lagrangian(
        W_est,
        rho,
        alpha
    )

    print(f"Objetivo:       {objective:.8f}")
    print(f"h(W):           {h:.8f}")
    print(f"rho:            {rho:.4f}")
    print(f"alpha:          {alpha:.8f}")

    # Critério de parada
    if abs(h) < 1e-8:
        print("\nConvergência atingida.")
        break

    # Atualização do multiplicador de Lagrange
    alpha += rho * h

    if abs(h) > 0.25 * h_previous:
        rho *= 10

    if rho > 10000:
        break

    h_previous = abs(h)


===== Iteração externa 1 =====
Objetivo:       1.48787744
h(W):           0.64439377
rho:            1.0000
alpha:          0.00000000

===== Iteração externa 2 =====
Objetivo:       1.84556526
h(W):           0.47389139
rho:            1.0000
alpha:          0.64439377

===== Iteração externa 3 =====
Objetivo:       2.18883030
h(W):           0.16549977
rho:            10.0000
alpha:          1.11828516

===== Iteração externa 4 =====
Objetivo:       2.60651677
h(W):           0.04517885
rho:            100.0000
alpha:          2.77328289

===== Iteração externa 5 =====
Objetivo:       2.78145020
h(W):           0.01220286
rho:            1000.0000
alpha:          7.29116806

===== Iteração externa 6 =====
Objetivo:       3.05204699
h(W):           0.00321870
rho:            10000.0000
alpha:          19.49403257


In [9]:
W_est[np.abs(W_est) < 0.3] = 0.0

In [10]:
W_est.reshape(dim, dim)

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.58612747,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.08881662,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        , -0.41509122,  0.        ,  0.        ,
         0.88064762],
       [ 0.35893682,  0.        ,  0.4709661 ,  0.        ,  0.        ,
         0.        ]])

In [11]:
# import numpy as np
# from scipy.linalg import expm

# from pymoo.core.problem import Problem
# from pymoo.algorithms.soo.nonconvex.ga import GA
# from pymoo.operators.crossover.sbx import SBX
# from pymoo.operators.mutation.pm import PM
# from pymoo.operators.sampling.rnd import FloatRandomSampling
# from pymoo.optimize import minimize

# # Assume-se que X (n x dim), n, dim e LAMBDA já estão definidos.

# # ============================================================
# # Parâmetros do Lagrangiano aumentado
# # ============================================================

# H_TOL = 1e-4          # (4) tolerância realista para um GA (antes: 1e-8)
# RHO_MAX = 1e6         # (4) teto para rho, evita mau condicionamento extremo
# MAX_OUTER = 10
# N_GEN = 1000
# POP_SIZE = 100
# WARM_SIGMA = 0.05     # (5) dispersão da população em torno da solução anterior
# XL, XU = -3.0, 3.0

# # ============================================================
# # (6) Diagonal fixada em zero: o GA busca só dim*dim - dim variáveis
# # ============================================================

# OFFDIAG = ~np.eye(dim, dtype=bool)
# N_VAR = int(OFFDIAG.sum())          # dim*dim - dim


# def vec_to_W(v):
#     """Monta W (dim x dim) com diagonal exatamente zero."""
#     W = np.zeros((dim, dim))
#     W[OFFDIAG] = v
#     return W


# def W_to_vec(W):
#     return W[OFFDIAG]


# # ============================================================
# # Restrição e objetivo
# # ============================================================

# def acyclic_restriction(W):
#     W = np.asarray(W).reshape(dim, dim)
#     return np.trace(expm(W * W)) - dim


# def augmented_lagrangian(W, rho, alpha):
#     W = np.asarray(W).reshape(dim, dim)

#     R = X - X @ W
#     reconstruction = 1 / (2 * n) * (R ** 2).sum()
#     l1_pen = LAMBDA * np.abs(W).sum()
#     h = acyclic_restriction(W)

#     return reconstruction + l1_pen + (rho / 2) * h ** 2 + alpha * h


# class NOTEARSLagrangianProblem(Problem):

#     def __init__(self, rho, alpha):
#         self.rho = rho
#         self.alpha = alpha
#         super().__init__(n_var=N_VAR, n_obj=1, xl=XL, xu=XU)

#     def _evaluate(self, V, out, *args, **kwargs):
#         F = np.array([
#             augmented_lagrangian(vec_to_W(v), self.rho, self.alpha)
#             for v in V
#         ])
#         out["F"] = F.reshape(-1, 1)


# def build_algorithm(x_prev=None, seed=0):
#     """(5) Se x_prev existe, a população inicial é centrada nele (warm start)."""
#     if x_prev is None:
#         sampling = FloatRandomSampling()
#     else:
#         rng = np.random.default_rng(seed)
#         pop0 = x_prev + WARM_SIGMA * rng.standard_normal((POP_SIZE, N_VAR))
#         pop0 = np.clip(pop0, XL, XU)
#         pop0[0] = x_prev            # preserva exatamente a solução anterior
#         sampling = pop0

#     return GA(
#         pop_size=POP_SIZE,
#         sampling=sampling,
#         crossover=SBX(prob=0.6, eta=15),
#         mutation=PM(prob=1.0 / N_VAR, eta=20),
#         eliminate_duplicates=True,
#     )


# # ============================================================
# # Laço externo
# # ============================================================

# rho = 1.0
# alpha = 0.0
# x_prev = None
# h_previous = np.inf
# W_est = None

# for outer_iter in range(MAX_OUTER):

#     print(f"\n===== Iteração externa {outer_iter + 1} =====")

#     problem = NOTEARSLagrangianProblem(rho=rho, alpha=alpha)
#     algorithm = build_algorithm(x_prev=x_prev, seed=42 + outer_iter)

#     res = minimize(
#         problem,
#         algorithm,
#         termination=("n_gen", N_GEN),
#         seed=42 + outer_iter,
#         verbose=False,
#     )

#     x_prev = res.X.copy()                 # (5) guarda para o próximo subproblema
#     W_est = vec_to_W(res.X)
#     h = acyclic_restriction(W_est)
#     objective = augmented_lagrangian(W_est, rho, alpha)

#     print(f"Objetivo:       {objective:.8f}")
#     print(f"h(W):           {h:.8e}")
#     print(f"rho:            {rho:.4e}")
#     print(f"alpha:          {alpha:.8f}")

#     # (4) critério de parada realista
#     if abs(h) < H_TOL:
#         print("\nConvergência atingida (h abaixo da tolerância).")
#         break

#     # Atualização do multiplicador de Lagrange
#     alpha += rho * h

#     # Aumenta rho só se a factibilidade não melhorou o bastante
#     if abs(h) > 0.25 * h_previous:
#         rho *= 10

#     h_previous = abs(h)

#     # (4) rho não pode crescer indefinidamente
#     if rho > RHO_MAX:
#         print("\nrho atingiu o teto; encerrando.")
#         break

# # Threshold final para ler a estrutura do grafo
# W_final = W_est.copy()
# W_final[np.abs(W_final) < 0.3] = 0.0
# print("\nh(W) após threshold:", acyclic_restriction(W_final))

In [12]:
# import numpy as np
# import math
# import time

# from scipy.linalg import expm
# from scipy.optimize import minimize


# # ============================================================
# # CONFIGURAÇÃO
# # ============================================================

# X = dataset.values.astype(float)

# n, d = X.shape

# lambda1 = 0.01

# max_iter = 100
# h_tol = 1e-8
# rho_max = 1e16

# rho = 1.0
# alpha = 0.0

# # Inicialização
# w = np.zeros(2 * d * d)


# # ============================================================
# # FUNÇÕES AUXILIARES
# # ============================================================

# def unpack(w):
#     """
#     Converte:
#         w = [w_plus, w_minus]

#     em:

#         W = w_plus - w_minus
#     """

#     w_plus = w[:d*d]
#     w_minus = w[d*d:]

#     W = (w_plus - w_minus).reshape(d, d)

#     return W


# def h_func(W):
#     """
#     Restrição de aciclicidade do NOTEARS:

#         h(W) = tr(exp(W o W)) - d

#     onde o é produto elemento-a-elemento.
#     """

#     E = expm(W * W)

#     h = np.trace(E) - d

#     return h, E


# def loss_and_grad(w, rho, alpha):
#     """
#     Augmented Lagrangian:

#         L(W) =
#             f(W)
#             + alpha * h(W)
#             + rho/2 * h(W)^2

#     onde:

#         f(W) =
#             1/(2n) ||X - XW||_F^2
#             + lambda1 ||W||_1
#     """

#     W = unpack(w)

#     # --------------------------------------------------------
#     # Reconstrução
#     # --------------------------------------------------------

#     R = X - X @ W

#     reconstruction_loss = (
#         0.5 / n
#         * np.sum(R ** 2)
#     )

#     # --------------------------------------------------------
#     # Aciclicidade
#     # --------------------------------------------------------

#     h, E = h_func(W)

#     # --------------------------------------------------------
#     # L1
#     #
#     # Como W = W+ - W-:
#     #
#     # ||W||_1 = sum(W+) + sum(W-)
#     # --------------------------------------------------------

#     w_plus = w[:d*d]
#     w_minus = w[d*d:]

#     l1_loss = lambda1 * np.sum(w_plus + w_minus)

#     # --------------------------------------------------------
#     # Augmented Lagrangian
#     # --------------------------------------------------------

#     loss = (
#         reconstruction_loss
#         + l1_loss
#         + alpha * h
#         + 0.5 * rho * h**2
#     )

#     # --------------------------------------------------------
#     # Gradiente da parte de reconstrução
#     #
#     # d/dW [1/(2n)||X-XW||²]
#     #
#     # = -Xᵀ(X-XW)/n
#     # --------------------------------------------------------

#     grad_W = (
#         -X.T @ R / n
#     )

#     # --------------------------------------------------------
#     # Gradiente da restrição
#     #
#     # dh/dW =
#     #
#     # 2 * E.T * W
#     #
#     # onde:
#     #
#     # E = exp(W o W)
#     # --------------------------------------------------------

#     grad_h = (
#         2 * E.T * W
#     )

#     # fator proveniente do Augmented Lagrangian
#     multiplier = (
#         alpha
#         + rho * h
#     )

#     grad_W += multiplier * grad_h

#     # --------------------------------------------------------
#     # Transformação para w+ e w-
#     #
#     # W = w+ - w-
#     #
#     # dL/dw+ = dL/dW + lambda
#     #
#     # dL/dw- = -dL/dW + lambda
#     # --------------------------------------------------------

#     grad_plus = (
#         grad_W.ravel()
#         + lambda1
#     )

#     grad_minus = (
#         -grad_W.ravel()
#         + lambda1
#     )

#     grad = np.concatenate([
#         grad_plus,
#         grad_minus
#     ])

#     return loss, grad


# # ============================================================
# # BOUNDS
# # ============================================================

# # w+ >= 0
# # w- >= 0
# #
# # diagonal = 0
# #
# # Isso evita que W tenha auto-loops.

# bounds = []

# for _ in range(2):

#     for i in range(d):

#         for j in range(d):

#             if i == j:
#                 bounds.append((0.0, 0.0))

#             else:
#                 bounds.append((0.0, None))


# # ============================================================
# # AUGMENTED LAGRANGIAN + L-BFGS-B
# # ============================================================

# start = time.perf_counter()

# h_prev = np.inf

# for iteration in range(max_iter):

#     # --------------------------------------------------------
#     # Otimização interna:
#     #
#     # min_w L(w, rho, alpha)
#     #
#     # usando L-BFGS-B
#     # --------------------------------------------------------

#     result = minimize(
#         fun=lambda w: loss_and_grad(
#             w,
#             rho,
#             alpha
#         ),

#         x0=w,

#         jac=True,

#         method="L-BFGS-B",

#         bounds=bounds,

#         options={
#             "maxiter": 1000,
#             "ftol": 1e-12,
#             "gtol": 1e-8,
#             "maxls": 50
#         }
#     )

#     # solução encontrada pelo L-BFGS
#     w = result.x

#     W = unpack(w)

#     # --------------------------------------------------------
#     # Calcula violação da restrição
#     # --------------------------------------------------------

#     h, _ = h_func(W)

#     # --------------------------------------------------------
#     # Atualiza multiplicador
#     #
#     # alpha_{k+1}
#     # =
#     # alpha_k + rho_k h(W_k)
#     # --------------------------------------------------------

#     alpha += rho * h

#     # --------------------------------------------------------
#     # Critério de convergência
#     # --------------------------------------------------------

#     if iteration == 0:

#         h_prev = abs(h)

#     else:

#         # Se a restrição não está diminuindo
#         # suficientemente, aumenta rho.

#         if abs(h) > 0.25 * h_prev:

#             rho *= 10.0

#         h_prev = abs(h)

#     # Limita rho
#     rho = min(rho, rho_max)

#     # --------------------------------------------------------
#     # Informações
#     # --------------------------------------------------------

#     print(
#         f"Iteração {iteration + 1:3d} | "
#         f"f(W) = {(
#             0.5 / n * np.sum(
#                 (X - X @ W) ** 2
#             )
#             + lambda1 * np.abs(W).sum()
#         ):12.6f} | "
#         f"h(W) = {h:12.4e} | "
#         f"rho = {rho:10.3e} | "
#         f"alpha = {alpha:10.3e} | "
#         f"L-BFGS: {result.success}"
#     )

#     # --------------------------------------------------------
#     # Convergência
#     # --------------------------------------------------------

#     if abs(h) <= h_tol:

#         print("\nConvergiu!")

#         break

# elapsed = time.perf_counter() - start


# # ============================================================
# # RESULTADO
# # ============================================================

# print("\n" + "=" * 60)
# print("RESULTADO")
# print("=" * 60)

# print(f"Tempo: {elapsed:.4f} s")
# print(f"Iterações externas: {iteration + 1}")
# print(f"h(W): {h:.4e}")
# print(f"rho: {rho:.4e}")
# print(f"alpha: {alpha:.4e}")
# print(f"L-BFGS success: {result.success}")
# print(f"Mensagem: {result.message}")

# print("\nW estimada:")
# print(np.round(W, 4))